# Data Import

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [ ]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [ ]:
train_data.head()

In [ ]:
train_data["dialogue"][0]

In [ ]:
train_data["summary"][0]

In [ ]:
train_data.shape

In [ ]:
val_data.head()

In [ ]:
val_data["dialogue"][0]

In [ ]:
val_data["summary"][0]

In [ ]:
val_data.shape

In [ ]:
# Here we are taking only the required data from the total dataset ie 4000 for training and 500 for validation
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [ ]:
train_data.shape

In [ ]:
val_data.shape

# Data Pre-Processing

In [ ]:
import re

def clean_data(text): # Here we have wrote a fnx which will clean our data using re library
    text = re.sub(r"\r\n", " ", text) # Lines 
    text = re.sub(r"\s+", " ", text) # Spaces
    text = re.sub(r"<.*?>", " ", text) # html tags <p> <h1>
    text = text.strip().lower()
    return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [ ]:
train_data["dialogue"][0]

In [ ]:
train_data["summary"][0]

In [ ]:
val_data["dialogue"][0]

In [ ]:
val_data["summary"][0]

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
# Take raw data => Tokenized i/p's for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"]
    return inputs

In [ ]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [ ]:
train_dataset[0]

In [ ]:
# input ids => dialogue ke tokens ye upper hee na wo
# 1 => EOS and 0 => padding
# attention mask = tells us which are valid token id values(1) and which are padding values(0)
# labels => summary ke tokens ye upper hee na wo

In [ ]:
len(train_dataset[0]["input_ids"])

In [ ]:
len(train_dataset[0]["labels"])

In [ ]:
len(train_dataset[0]["attention_mask"])

# Working with our model

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("device: ", device)
model.to(device)

In [ ]:
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Train the model

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

# Test the core logic for summarization

In [ ]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token id's
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # decode or convert the token id's to summary => decoding
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # skipping EOS and SEP special tokens
    return summary

In [ ]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)